# Lab 14 — 없는 효과 만들어내기

**확률통계 · Topic 14 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. **효과가 0인 데이터**로 "유의한" 결과를 만들어낸다 (직접 p-hacking을 저질러 본다).
2. 다중비교 보정으로 되돌려본다.
3. **permutation test**를 직접 구현하고 t-test와 비교한다.

⏱ **예상 소요 시간: 35분**

> ⚠️ 오늘 배우는 것은 **"하지 말아야 할 것"** 이다.
> 직접 해봐야 남의 분석에서 알아볼 수 있다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)
ALPHA = 0.05
print("준비 완료")

## Part 1. 정직한 A/B 테스트

먼저 **효과가 진짜로 0인** 두 그룹을 만든다. 정직하게 **한 번만** 검정한다.

### 실습 1 — 위양성률 확인

In [ ]:
def one_test(n=100, effect=0.0, rng=rng):
    a = rng.normal(0, 1, n)
    b = rng.normal(effect, 1, n)
    return stats.ttest_ind(a, b).pvalue


pvals = [one_test() for _ in range(5000)]

# TODO 1: p < ALPHA 인 비율(위양성률)을 구하세요
false_positive_rate = 0.0

print(f"효과가 0인데 '유의'하다고 판정한 비율: {false_positive_rate:.4f}")
print(f"의도한 유의수준 alpha: {ALPHA}")

plt.figure(figsize=(6.5, 3.6))
plt.hist(pvals, bins=40)
plt.axvline(ALPHA, color="red", ls="--", lw=2)
plt.xlabel("p-value")
plt.ylabel("count")
plt.title("p-values under H0 are uniform")
plt.show()

> 위양성률이 약 5% — **의도한 대로다.** 그리고 p-value의 히스토그램이 **평평하다.**
> 귀무가설이 참이면 p-value는 $U(0,1)$ 을 따른다. 이것이 정상 상태다.

## Part 2. 죄를 지어보자 (1) — 지표를 여러 개 보기

"전환율, 체류시간, 클릭수, 재방문율, 만족도, ..." 지표를 여러 개 보면 어떻게 될까?

### 실습 2

In [ ]:
def any_significant(k, n=100, rng=rng):
    # TODO 2: k개의 검정 중 하나라도 p < ALPHA 이면 True를 돌려주세요
    #         힌트: any(one_test(n, 0.0, rng) < ALPHA for _ in range(k))
    return False


ks = [1, 2, 5, 10, 20, 40]
rates = [np.mean([any_significant(k) for _ in range(1000)]) for k in ks]

print(f"{'지표 수':>8}{'위양성률':>12}{'이론값':>12}")
for k, r in zip(ks, rates):
    print(f"{k:>8}{r:>12.4f}{1 - (1 - ALPHA) ** k:>12.4f}")

😈 **지표 20개만 봐도 "무언가 유의한 것"을 찾을 확률이 60%를 넘는다.**

논문이나 보고서에서 "여러 지표 중 ○○에서 유의한 차이를 발견했다"는 문장을 보면
**몇 개를 봤는지** 물어야 하는 이유다.

## Part 3. 죄를 지어보자 (2) — 유의해질 때까지 데이터 모으기

"아직 유의하지 않네, 100명만 더 모아보자." 이것이 왜 문제일까?

### 실습 3

In [ ]:
def peeking(n_max=500, step=20, rng=rng):
    a = rng.normal(0, 1, n_max)
    b = rng.normal(0, 1, n_max)
    for m in range(step, n_max + 1, step):
        # TODO 3: 이 시점에서 p < ALPHA 이면 True를 돌려주세요
        pass
    return False


rate_peek = np.mean([peeking() for _ in range(2000)])

print(f"중간에 계속 들여다봤을 때 위양성률: {rate_peek:.4f}")
print(f"한 번만 검정했을 때:              {ALPHA}")

😈 **데이터를 들여다보는 행위 자체가 검정을 여러 번 하는 것**이다.

이것을 **optional stopping** 또는 **peeking** 이라고 한다.
실무 A/B 테스트에서 가장 흔한 실수이며, 대부분 **악의 없이** 일어난다.

> 막는 방법: **표본 크기를 미리 정하고 끝까지 간다.**
> 중간에 봐야 한다면 그에 맞는 방법(순차 검정)을 써야 한다.

## Part 4. 되돌리기 — Bonferroni 보정

$k$개를 검정한다면 각 검정의 기준을 $\alpha/k$ 로 낮춘다.

### 실습 4

In [ ]:
def any_significant_bonferroni(k, n=100, rng=rng):
    # TODO 4: 유의수준을 k로 나눈 뒤 검정하세요
    adjusted = ALPHA
    return any(one_test(n, 0.0, rng) < adjusted for _ in range(k))


print(f"{'지표 수':>8}{'보정 전':>12}{'보정 후':>12}")
for k in [1, 5, 10, 20, 40]:
    before = np.mean([any_significant(k) for _ in range(1000)])
    after = np.mean([any_significant_bonferroni(k) for _ in range(1000)])
    print(f"{k:>8}{before:>12.4f}{after:>12.4f}")

✅ **보정하면 위양성률이 다시 5% 근처로 돌아온다.**

⚠️ 단, 공짜가 아니다. 기준이 엄격해진 만큼 **진짜 효과도 놓치기 쉬워진다**(검정력 감소).
더 나은 방법(FDR)은 확률통계 II에서 배운다.

## Part 5. Permutation test 직접 만들기

오늘 강의 [S]에서 한 것을 직접 구현한다. **t-test와 비교**해보자.

### 실습 5

In [ ]:
def permutation_test(a, b, n_perm=10000, rng=rng):
    observed = b.mean() - a.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)

    diffs = np.empty(n_perm)
    for i in range(n_perm):
        s = rng.permutation(pooled)
        diffs[i] = s[n_a:].mean() - s[:n_a].mean()

    # TODO 5: 관측값만큼 극단적인 경우의 비율(= p-value)을 구하세요
    #         힌트: (np.abs(diffs) >= abs(observed)).mean()
    p = 1.0
    return observed, p, diffs


group_a = rng.normal(52.0, 3.0, 25)
group_b = rng.normal(54.2, 3.0, 25)

obs, p_perm, diffs = permutation_test(group_a, group_b)
p_t = stats.ttest_ind(group_a, group_b).pvalue

print(f"관측된 차이     : {obs:.4f}")
print(f"permutation p   : {p_perm:.4f}")
print(f"t-test p        : {p_t:.4f}")

plt.figure(figsize=(7, 3.8))
plt.hist(diffs, bins=60, alpha=0.85)
plt.axvline(obs, color="red", lw=2.5)
plt.axvline(-obs, color="red", lw=2.5, ls="--")
plt.xlabel("difference after shuffling labels")
plt.ylabel("count")
plt.title("Permutation distribution")
plt.show()

> **두 p-value가 거의 같다.** 데이터가 정규분포에 가까우면 t-test는 좋은 근사다.
> 하지만 permutation은 **정규성을 가정하지 않는다.**
> 중앙값이든 90 퍼센타일이든 **어떤 통계량에도** 쓸 수 있다는 것이 진짜 장점이다.

---

## 마무리 — 자가 점검

- [ ] 귀무가설이 참이면 p-value가 균등분포임을 확인했다
- [ ] 지표를 늘리면 위양성이 폭증하는 것을 직접 만들었다
- [ ] 중간에 들여다보는 것이 왜 문제인지 설명할 수 있다
- [ ] Bonferroni 보정을 적용할 수 있다
- [ ] permutation test를 직접 구현했다

**오늘 배운 함정 중 자신이 저지를 뻔했던 것을 하나 쓰자.**

> (여기에 작성)

### 📌 미니 프로젝트 2의 ⑤번(내 분석이 틀릴 수 있는 이유)을 오늘 목록으로 점검할 것